<a href="https://colab.research.google.com/github/claudiohenriquezberroeta-pucv/Tarea-2---Desafio-Kaggle/blob/main/efficientnet_b3_de_kaggle_food_colab_checkpoints.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clasificación Jerárquica de Comidas — Versión COLAB con Checkpoints

## Sistema de recuperación ante interrupciones de Colab

**Problema:** Colab desconecta antes de terminar el entrenamiento.

**Solución:** Checkpoint completo en Google Drive cada época. Al re-ejecutar, el notebook:
1. Detecta si existe un checkpoint en Drive
2. Restaura modelo, optimizador, scheduler e historial
3. Continúa desde la época siguiente sin perder nada

**Escalable:** Cambia `CFG['backbone']` para probar múltiples modelos preentrenados.


In [ ]:
# ─── Celda 1: Instalar dependencias ─────────────────────────────────────────
!pip install -q timm torchmetrics albumentations
print('Dependencias instaladas')

Dependencias instaladas


In [ ]:
# ─── Celda 2: Montar Google Drive (CRÍTICO para persistencia) ─────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
# Directorio base en Drive — aquí se guardan todos los checkpoints
DRIVE_DIR = '/content/drive/MyDrive/kaggle_food101'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive montado. Checkpoints en: {DRIVE_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive montado. Checkpoints en: /content/drive/MyDrive/kaggle_food101


In [ ]:
# ─── Celda 3: Imports y configuración GPU ────────────────────────────────────
import os
import random
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, OneCycleLR
from torch.cuda.amp import GradScaler, autocast

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
import warnings
warnings.filterwarnings('ignore')

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # True si img_size fijo siempre
set_seed()

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    # Habilitar benchmark para GPU fija
    torch.backends.cudnn.benchmark = True

Dispositivo: cuda
GPU: Tesla T4
VRAM: 15.6 GB


In [ ]:
# Descomprimiento zip de archivos en content de Colab
# ideal para reducir tiempo de lectura de cada archivo desde drive

from pathlib import Path

# Opción 1: con Path (recomendado)
ruta_actual = Path.cwd()
print(ruta_actual)

# Opción 2: con os
ruta_actual_os = os.getcwd()
print(ruta_actual_os)

!unzip -q "/content/drive/MyDrive/kaggle_food101/clasificacion-jerarquica-de-comidas-del-mundo.zip" -d /content/food_data



/content
/content
replace /content/food_data/kaggle_food101/dict.npy? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
CFG = {
    # ── MODELO (cambia esto para escalar a otros backbones) ──
    'backbone'       : 'efficientnet_b3',  # ← CAMBIAR AQUÍ

    # ── IMÁGENES ──
    'img_size'       : 224,   # Aumentar a 384 para swin/vit si tienes GPU Pro

    # ── DATALOADER ──
    'batch_size'     : 64,    # T4: 64 | V100/A100: 128
    'num_workers'    : 4,
    'pin_memory'     : True,

    # ── ENTRENAMIENTO ──
    'num_epochs'     : 30,
    'lr'             : 3e-4,
    'weight_decay'   : 1e-4,
    'label_smoothing': 0.1,
    'mixup_alpha'    : 0.4,
    'mixup_prob'     : 0.5,
    'n_folds'        : 5,
    'fold'           : 0,
    'patience'       : 7,     # Early stopping

    # ── CHECKPOINT (NO MODIFICAR) ──
    'checkpoint_every': 1,    # Guardar en Drive cada N épocas (1 = cada época)
}

# Rutas del dataset en Colab
# Opción A: Kaggle API (recomendado)
#!kaggle competitions download -c 'clasificacion-jerarquica-de-comidas-del-mundo'

# Con datos cargados en content
DATA_DIR  = Path('/content/food_data/kaggle_food101')  # Ruta de datos descomprimidos en content (colab)
TRAIN_DIR = DATA_DIR / 'train'
TEST_DIR  = DATA_DIR / 'test'

train_df   = pd.read_csv(DATA_DIR / 'train.csv')
test_df    = pd.read_csv(DATA_DIR / 'test_public.csv')
label_dict = np.load(DATA_DIR / 'dict.npy', allow_pickle=True).item()

train_df[['label_l1', 'label_l2']] = train_df['label'].str.split(' ', expand=True).astype(int)
L1_NAMES = {int(k): v for k, v in label_dict.items() if int(k) < 6}
L2_NAMES = {int(k): v for k, v in label_dict.items() if int(k) >= 6}

print(f'Train: {len(train_df)} | Test: {len(test_df)}')
print(f'Backbone: {CFG["backbone"]} | img_size: {CFG["img_size"]} | batch: {CFG["batch_size"]}')

In [ ]:
# Parsear etiquetas jerárquicas
train_df[['label_l1', 'label_l2']] = train_df['label'].str.split(' ', expand=True).astype(int)

# Mapas de etiquetas a nombres
L1_NAMES = {int(k): v for k, v in label_dict.items() if int(k) < 6}
L2_NAMES = {int(k): v for k, v in label_dict.items() if int(k) >= 6}

print('Clases Nivel 1 (Origen Geográfico):')
for k, v in L1_NAMES.items():
    print(f'  {k}: {v}')

print(f'\nClases Nivel 2 (Platos específicos): {len(L2_NAMES)} clases')

In [ ]:
# ─── Distribución de Nivel 1 ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

l1_counts = train_df['label_l1'].value_counts().sort_index()
l1_names  = [L1_NAMES[i] for i in l1_counts.index]

bars = axes[0].bar(l1_names, l1_counts.values, color=plt.cm.Set2.colors[:6], edgecolor='black', linewidth=0.5)
axes[0].set_title('Distribución de Clases - Nivel 1 (Origen Geográfico)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Categoría')
axes[0].set_ylabel('Número de imágenes')
axes[0].tick_params(axis='x', rotation=25)
for bar, val in zip(bars, l1_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(val), ha='center', va='bottom', fontsize=9)

# Calcular desbalance
imbalance_ratio = l1_counts.max() / l1_counts.min()
axes[0].set_title(f'Distribución Nivel 1 | Imbalance ratio: {imbalance_ratio:.2f}x', fontsize=12, fontweight='bold')

# Pie chart
axes[1].pie(l1_counts.values, labels=l1_names, autopct='%1.1f%%',
            colors=plt.cm.Set2.colors[:6], startangle=90)
axes[1].set_title('Proporción por Origen Geográfico', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('eda_nivel1.png', dpi=150, bbox_inches='tight')
plt.show()
print(l1_counts.to_frame('count').rename(index=L1_NAMES))

In [ ]:
# ─── Distribución de Nivel 2 ───────────────────────────────────────────────
l2_counts = train_df['label_l2'].value_counts().sort_index()

plt.figure(figsize=(22, 5))
l2_names_sorted = [L2_NAMES[i] for i in l2_counts.index]
plt.bar(range(len(l2_counts)), l2_counts.values, color='steelblue', alpha=0.8)
plt.xticks(range(len(l2_counts)), l2_names_sorted, rotation=90, fontsize=7)
plt.axhline(l2_counts.mean(), color='red', linestyle='--', label=f'Media: {l2_counts.mean():.0f}')
plt.title('Distribución de Clases - Nivel 2 (Plato Específico)', fontsize=13, fontweight='bold')
plt.ylabel('Número de imágenes')
plt.legend()
plt.tight_layout()
plt.savefig('eda_nivel2.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Min imágenes por clase L2: {l2_counts.min()} ({L2_NAMES[l2_counts.idxmin()]})')
print(f'Max imágenes por clase L2: {l2_counts.max()} ({L2_NAMES[l2_counts.idxmax()]})')
print(f'Media: {l2_counts.mean():.1f} | Std: {l2_counts.std():.1f}')

In [ ]:
# ─── Visualización de ejemplos por categoría ────────────────────────────────
def show_category_examples(train_df, train_dir, label_dict, n_categories=6, n_per_cat=4):
    """Muestra ejemplos de imágenes por categoría de nivel 1."""
    fig, axes = plt.subplots(n_categories, n_per_cat, figsize=(n_per_cat*3, n_categories*3))

    for row, cat_id in enumerate(range(n_categories)):
        cat_df = train_df[train_df['label_l1'] == cat_id]
        samples = cat_df.sample(min(n_per_cat, len(cat_df)), random_state=42)

        for col, (_, sample) in enumerate(samples.iterrows()):
            img_path = train_dir / sample['filename']
            img = Image.open(img_path).convert('RGB')
            axes[row, col].imshow(img)
            axes[row, col].axis('off')
            if col == 0:
                axes[row, col].set_ylabel(L1_NAMES[cat_id], fontsize=10, fontweight='bold')
            axes[row, col].set_title(L2_NAMES[sample['label_l2']], fontsize=7)

    plt.suptitle('Ejemplos por Categoría Geográfica', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('ejemplos_categorias.png', dpi=120, bbox_inches='tight')
    plt.show()

show_category_examples(train_df, TRAIN_DIR, label_dict)

In [ ]:
# ─── Celda 5: Sistema de checkpoints ─────────────────────────────────────────

def get_checkpoint_path(fold=CFG['fold'], backbone=CFG['backbone']):
    """Ruta única por backbone+fold para soportar múltiples experimentos."""
    name = backbone.replace('/', '_')
    return os.path.join(DRIVE_DIR, f'ckpt_{name}_fold{fold}.pt')


def save_checkpoint(epoch, model, optimizer, scheduler, history, best_f1,
                    best_epoch, patience_counter, fold=CFG['fold']):
    """
    Guarda estado completo en Google Drive.
    Incluye modelo, optimizador, scheduler, historial y metadatos.
    """
    ckpt_path = get_checkpoint_path(fold)
    checkpoint = {
        # Metadatos
        'epoch'           : epoch,
        'backbone'        : CFG['backbone'],
        'img_size'        : CFG['img_size'],
        'fold'            : fold,
        'best_f1'         : best_f1,
        'best_epoch'      : best_epoch,
        'patience_counter': patience_counter,
        'timestamp'       : time.strftime('%Y-%m-%d %H:%M:%S'),
        # Estados
        'model_state'     : model.state_dict(),
        'optimizer_state' : optimizer.state_dict(),
        'scheduler_state' : scheduler.state_dict() if scheduler else None,
        'history'         : history,
        'cfg'             : CFG,
    }
    torch.save(checkpoint, ckpt_path)
    print(f'Checkpoint guardado: época {epoch} | F1: {best_f1:.4f} → {ckpt_path}')


def load_checkpoint(model, optimizer, scheduler, fold=CFG['fold']):
    """
    Carga checkpoint si existe.
    NOTA: Solo carga el modelo aquí. El estado del optimizador se
    carga después de reconstruir el optimizador correcto según la fase.
    """
    ckpt_path = get_checkpoint_path(fold)

    if not os.path.exists(ckpt_path):
        print('No se encontró checkpoint. Iniciando desde cero.')
        return 1, {'train_loss': [], 'val_loss': [], 'f1_l1': [], 'f1_l2': [], 'f1_comb': []}, \
               0.0, 0, 0

    print(f'Checkpoint encontrado: {ckpt_path}')
    ckpt = torch.load(ckpt_path, map_location=DEVICE)

    if ckpt.get('backbone') != CFG['backbone']:
        print(f'Backbone en checkpoint ({ckpt["backbone"]}) != CFG ({CFG["backbone"]})')
        print('   Iniciando desde cero con nuevo backbone.')
        return 1, {'train_loss': [], 'val_loss': [], 'f1_l1': [], 'f1_l2': [], 'f1_comb': []}, \
               0.0, 0, 0

    # Solo cargamos el modelo — el optimizador se carga después
    model.load_state_dict(ckpt['model_state'])

    # NO llamamos optimizer.load_state_dict() aquí

    start_epoch      = ckpt['epoch'] + 1
    history          = ckpt['history']
    best_f1          = ckpt['best_f1']
    best_epoch       = ckpt['best_epoch']
    patience_counter = ckpt.get('patience_counter', 0)

    print(f'   Reanudando desde época {start_epoch} | '
          f'Mejor F1: {best_f1:.4f} (época {best_epoch}) | '
          f'Guardado: {ckpt["timestamp"]}')
    return start_epoch, history, best_f1, best_epoch, patience_counter

def save_best_model_separately(model, fold=CFG['fold']):
    """Guarda solo el mejor modelo (sin optimizador) para inference."""
    name = CFG['backbone'].replace('/', '_')
    path = os.path.join(DRIVE_DIR, f'best_model_{name}_fold{fold}.pth')
    torch.save(model.state_dict(), path)
    return path


print('Sistema de checkpoints definido')

In [ ]:
# ─── Celda 6: Transforms y Dataset ───────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def get_train_transforms(img_size=CFG['img_size']):
    return A.Compose([
        A.RandomResizedCrop((img_size, img_size), scale=(0.7, 1.0), ratio=(0.75, 1.33)),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=15, p=0.5),
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2),
            A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30),
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        ], p=0.7),
        A.OneOf([
            A.GaussianBlur(blur_limit=3),
            A.MotionBlur(blur_limit=3),
            A.GaussNoise(var_limit=(10, 50)),
        ], p=0.3),
        A.CoarseDropout(max_holes=8, max_height=img_size//8, max_width=img_size//8,
                        min_holes=1, fill_value=0, p=0.3),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def get_val_transforms(img_size=CFG['img_size']):
    return A.Compose([
        A.Resize(int(img_size * 1.1), int(img_size * 1.1)),
        A.CenterCrop(img_size, img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


class FoodDataset(Dataset):
    def __init__(self, df, img_dir, transforms=None, is_test=False):
        self.df         = df.reset_index(drop=True)
        self.img_dir    = img_dir
        self.transforms = transforms
        self.is_test    = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = np.array(Image.open(self.img_dir / row['filename']).convert('RGB'))
        if self.transforms:
            img = self.transforms(image=img)['image']
        if self.is_test:
            return img, row['filename']
        l1 = int(row['label_l1'])
        l2 = int(row['label_l2']) - 6
        return img, l1, l2


def make_weighted_sampler(df):
    counts  = df['label_l2'].value_counts()
    weights = df['label_l2'].map(lambda x: 1.0 / counts[x]).values
    return torch.utils.data.WeightedRandomSampler(
        weights=torch.FloatTensor(weights),
        num_samples=len(weights),
        replacement=True
    )

print('Transforms y Dataset definidos')

In [ ]:
import huggingface_hub
huggingface_hub.login()

In [ ]:
# ─── Celda 7: Modelo Jerárquico (escalable para cualquier backbone timm) ──────
class HierarchicalFoodClassifier(nn.Module):
    def __init__(self, backbone_name=CFG['backbone'], num_l1=6, num_l2=101,
                 pretrained=True, dropout=0.3):
        super().__init__()
        self.backbone_name = backbone_name

        # timm soporta +600 backbones con la misma interfaz
        self.backbone = timm.create_model(
            backbone_name,
            pretrained=pretrained,
            num_classes=0,
            global_pool='avg'
        )
        feat_dim = self.backbone.num_features

        # Neck adaptativo según feat_dim del backbone
        neck_dim = min(1024, max(256, feat_dim // 2))

        self.neck = nn.Sequential(
            nn.Linear(feat_dim, neck_dim),
            nn.BatchNorm1d(neck_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.head_l1 = nn.Sequential(
            nn.Linear(neck_dim, neck_dim // 4),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(neck_dim // 4, num_l1)
        )
        self.head_l2 = nn.Sequential(
            nn.Linear(neck_dim + num_l1, neck_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(neck_dim // 2, num_l2)
        )
        for head in [self.head_l1, self.head_l2, self.neck]:
            for m in head.modules():
                if isinstance(m, nn.Linear):
                    nn.init.xavier_uniform_(m.weight)
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)

    def forward(self, x):
        feats     = self.backbone(x)
        shared    = self.neck(feats)
        logits_l1 = self.head_l1(shared)
        l1_soft   = F.softmax(logits_l1.detach(), dim=-1)
        l2_input  = torch.cat([shared, l1_soft], dim=-1)
        logits_l2 = self.head_l2(l2_input)
        return logits_l1, logits_l2


# Verificar backbone
model = HierarchicalFoodClassifier(backbone_name=CFG['backbone']).to(DEVICE)
dummy = torch.randn(2, 3, CFG['img_size'], CFG['img_size']).to(DEVICE)
with torch.no_grad():
    o1, o2 = model(dummy)
total_params = sum(p.numel() for p in model.parameters())
print(f'Backbone: {CFG["backbone"]} | feat_dim: {model.backbone.num_features}')
print(f'   L1: {o1.shape} | L2: {o2.shape} | Params: {total_params:,}')

In [ ]:
# ─── Celda 8: Pérdidas, Mixup y métricas ─────────────────────────────────────
class HierarchicalLoss(nn.Module):
    def __init__(self, alpha=0.3, label_smoothing=0.1):
        super().__init__()
        self.alpha = alpha
        self.ce_l1 = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
        self.ce_l2 = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    def forward(self, logits_l1, logits_l2, targets_l1, targets_l2):
        loss_l1 = self.ce_l1(logits_l1, targets_l1)
        loss_l2 = self.ce_l2(logits_l2, targets_l2)
        return self.alpha * loss_l1 + (1 - self.alpha) * loss_l2, loss_l1, loss_l2


def mixup_data(x, y_l1, y_l2, alpha=0.4):
    lam   = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    index = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[index], y_l1, y_l1[index], y_l2, y_l2[index], lam


def mixup_criterion(criterion, logits_l1, logits_l2, y_l1a, y_l1b, y_l2a, y_l2b, lam):
    loss_a, _, _ = criterion(logits_l1, logits_l2, y_l1a, y_l2a)
    loss_b, _, _ = criterion(logits_l1, logits_l2, y_l1b, y_l2b)
    return lam * loss_a + (1 - lam) * loss_b


def hierarchical_f1(pred_l1, pred_l2, true_l1, true_l2):
    f1_l1       = f1_score(true_l1, pred_l1, average='macro', zero_division=0)
    f1_l2       = f1_score(true_l2, pred_l2, average='macro', zero_division=0)
    all_preds   = np.concatenate([pred_l1, pred_l2 + 6])
    all_trues   = np.concatenate([true_l1, true_l2 + 6])
    f1_combined = f1_score(all_trues, all_preds, average='macro', zero_division=0)
    return f1_l1, f1_l2, f1_combined


print('Funciones de pérdida y métricas definidas')

In [ ]:
# ─── Celda 9: Fine-tuning progresivo ─────────────────────────────────────────
def freeze_backbone(model):
    for param in model.backbone.parameters():
        param.requires_grad = False

def unfreeze_last_n_blocks(model, n=3):
    blocks = list(model.backbone.blocks.children()) if hasattr(model.backbone, 'blocks') \
             else list(model.backbone.children())
    for block in blocks[-n:]:
        for param in block.parameters():
            param.requires_grad = True

def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True

print('Funciones de fine-tuning definidas')

In [ ]:
# ─── Celda 10: Loops de entrenamiento con AMP (GPU) ──────────────────────────
def train_one_epoch(model, loader, optimizer, criterion, scaler, epoch):
    model.train()
    total_loss = 0
    t0 = time.time()

    for batch_idx, (imgs, l1, l2) in enumerate(loader):
        imgs = imgs.to(DEVICE, non_blocking=True)
        l1   = l1.to(DEVICE)
        l2   = l2.to(DEVICE)

        use_mixup = random.random() < CFG['mixup_prob']
        if use_mixup:
            imgs, l1a, l1b, l2a, l2b, lam = mixup_data(imgs, l1, l2, CFG['mixup_alpha'])

        optimizer.zero_grad()

        with autocast():
            logits_l1, logits_l2 = model(imgs)
            if use_mixup:
                loss = mixup_criterion(criterion, logits_l1, logits_l2, l1a, l1b, l2a, l2b, lam)
            else:
                loss, _, _ = criterion(logits_l1, logits_l2, l1, l2)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

        if (batch_idx + 1) % 20 == 0:
            elapsed = time.time() - t0
            eta = elapsed / (batch_idx + 1) * (len(loader) - batch_idx - 1)
            print(f'  [{batch_idx+1}/{len(loader)}] Loss: {loss.item():.4f} | ETA: {eta:.0f}s',
                  end='\r', flush=True)

    print()
    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_pred_l1, all_pred_l2 = [], []
    all_true_l1, all_true_l2 = [], []

    for imgs, l1, l2 in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        l1   = l1.to(DEVICE)
        l2   = l2.to(DEVICE)
        logits_l1, logits_l2 = model(imgs)
        loss, _, _ = criterion(logits_l1, logits_l2, l1, l2)
        total_loss   += loss.item()
        all_pred_l1.extend(logits_l1.argmax(1).cpu().numpy())
        all_pred_l2.extend(logits_l2.argmax(1).cpu().numpy())
        all_true_l1.extend(l1.cpu().numpy())
        all_true_l2.extend(l2.cpu().numpy())

    f1_l1, f1_l2, f1_comb = hierarchical_f1(
        np.array(all_pred_l1), np.array(all_pred_l2),
        np.array(all_true_l1), np.array(all_true_l2)
    )
    return total_loss / len(loader), f1_l1, f1_l2, f1_comb


print('Loops de entrenamiento/validación definidos')

In [ ]:
# ─── Celda 11: Función principal con checkpoint auto-resume ───────────────────
def train_model(train_df, fold=CFG['fold']):
    # Split train/val
    skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=SEED)
    for i, (trn_idx, val_idx) in enumerate(skf.split(train_df, train_df['label_l1'])):
        if i == fold:
            break

    trn_df = train_df.iloc[trn_idx].reset_index(drop=True)
    val_df = train_df.iloc[val_idx].reset_index(drop=True)
    print(f'Fold {fold}: Train={len(trn_df)} | Val={len(val_df)}')

    trn_ds = FoodDataset(trn_df, TRAIN_DIR, get_train_transforms())
    val_ds = FoodDataset(val_df, TRAIN_DIR, get_val_transforms())

    trn_sampler = make_weighted_sampler(trn_df)
    trn_loader  = DataLoader(trn_ds, batch_size=CFG['batch_size'], sampler=trn_sampler,
                             num_workers=CFG['num_workers'], pin_memory=CFG['pin_memory'])
    val_loader  = DataLoader(val_ds, batch_size=CFG['batch_size'] * 2, shuffle=False,
                             num_workers=CFG['num_workers'], pin_memory=CFG['pin_memory'])

    # Modelo y criterio
    model     = HierarchicalFoodClassifier(backbone_name=CFG['backbone']).to(DEVICE)
    criterion = HierarchicalLoss(alpha=0.3, label_smoothing=CFG['label_smoothing'])
    scaler    = GradScaler()

    # Optimizador y scheduler iniciales (pueden sobreescribirse por checkpoint)
    freeze_backbone(model)
    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                      lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    scheduler = OneCycleLR(optimizer, max_lr=CFG['lr'],
                           steps_per_epoch=len(trn_loader), epochs=5)

    # ── AUTO-RESUME: cargar checkpoint si existe ──────────────────────────────
    start_epoch, history, best_f1, best_epoch, patience_counter = \
        load_checkpoint(model, optimizer, scheduler, fold)

    # Reconstruir el estado de fine-tuning según start_epoch
    if start_epoch > 16:
        print('  → Restaurando Fase 3 (todo descongelado)')
        unfreeze_all(model)
        optimizer = AdamW(model.parameters(),
                          lr=CFG['lr'] / 50, weight_decay=CFG['weight_decay'])
        scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)
        # Recargar estados del checkpoint restaurado
        ckpt_path = get_checkpoint_path(fold)
        if os.path.exists(ckpt_path):
            ckpt = torch.load(ckpt_path, map_location=DEVICE)
            optimizer.load_state_dict(ckpt['optimizer_state'])
            if ckpt.get('scheduler_state'):
                scheduler.load_state_dict(ckpt['scheduler_state'])
    elif start_epoch > 6:
        print('  → Restaurando Fase 2 (últimas capas descongeladas)')
        unfreeze_last_n_blocks(model, n=3)
        optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                          lr=CFG['lr'] / 10, weight_decay=CFG['weight_decay'])
        scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=1)
        ckpt_path = get_checkpoint_path(fold)
        if os.path.exists(ckpt_path):
            ckpt = torch.load(ckpt_path, map_location=DEVICE)
            optimizer.load_state_dict(ckpt['optimizer_state'])
            if ckpt.get('scheduler_state'):
                scheduler.load_state_dict(ckpt['scheduler_state'])

    if start_epoch > CFG['num_epochs']:
        print(f'Entrenamiento ya completado ({CFG["num_epochs"]} épocas). F1: {best_f1:.4f}')
        model.load_state_dict(torch.load(
            os.path.join(DRIVE_DIR, f'best_model_{CFG["backbone"].replace("/","_")}_fold{fold}.pth'),
            map_location=DEVICE
        ))
        return model, history, best_f1

    print(f'\nIniciando/reanudando entrenamiento desde época {start_epoch}...\n')

    for epoch in range(start_epoch, CFG['num_epochs'] + 1):
        t_epoch = time.time()

        # Transiciones de fase
        if epoch == 6 and start_epoch <= 6:
            print('\n→ Fase 2: Descongelando últimas 3 capas del backbone')
            unfreeze_last_n_blocks(model, n=3)
            optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                              lr=CFG['lr'] / 10, weight_decay=CFG['weight_decay'])
            scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=1)

        if epoch == 16 and start_epoch <= 16:
            print('\n→ Fase 3: Descongelando todo el modelo')
            unfreeze_all(model)
            optimizer = AdamW(model.parameters(),
                              lr=CFG['lr'] / 50, weight_decay=CFG['weight_decay'])
            scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)

        # Entrenamiento y validación
        train_loss = train_one_epoch(model, trn_loader, optimizer, criterion, scaler, epoch)
        val_loss, f1_l1, f1_l2, f1_comb = validate(model, val_loader, criterion)
        scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['f1_l1'].append(f1_l1)
        history['f1_l2'].append(f1_l2)
        history['f1_comb'].append(f1_comb)

        epoch_time = time.time() - t_epoch
        is_best = f1_comb > best_f1

        if is_best:
            best_f1    = f1_comb
            best_epoch = epoch
            patience_counter = 0
            best_path = save_best_model_separately(model, fold)
            marker = f' ← Mejor! ({best_path})'
        else:
            patience_counter += 1
            marker = ''

        print(f'Época {epoch:02d}/{CFG["num_epochs"]} [{epoch_time:.0f}s] | '
              f'Loss T:{train_loss:.4f} V:{val_loss:.4f} | '
              f'F1 L1:{f1_l1:.4f} L2:{f1_l2:.4f} Comb:{f1_comb:.4f}'
              f'{marker}')

        # Guardar checkpoint en Drive
        if epoch % CFG['checkpoint_every'] == 0:
            save_checkpoint(epoch, model, optimizer, scheduler, history,
                            best_f1, best_epoch, patience_counter, fold)

        # Early stopping
        if patience_counter >= CFG['patience']:
            print(f'\n Early stopping: sin mejora en {CFG["patience"]} épocas.')
            # Guardar checkpoint final antes de parar
            save_checkpoint(epoch, model, optimizer, scheduler, history,
                            best_f1, best_epoch, patience_counter, fold)
            break

    print(f'\nMejor época: {best_epoch} | Mejor F1 combinado: {best_f1:.4f}')

    # Cargar el mejor modelo
    name = CFG['backbone'].replace('/', '_')
    best_path = os.path.join(DRIVE_DIR, f'best_model_{name}_fold{fold}.pth')
    if os.path.exists(best_path):
        model.load_state_dict(torch.load(best_path, map_location=DEVICE))

    return model, history, best_f1


# ── EJECUTAR ──
model, history, best_f1 = train_model(train_df, fold=CFG['fold'])

In [ ]:
# ─── Celda 12: Visualizar curvas de aprendizaje ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_range, history['train_loss'], 'b-o', markersize=4, label='Train Loss')
axes[0].plot(epochs_range, history['val_loss'],   'r-o', markersize=4, label='Val Loss')
for xline, ls, label in [(6, '--', 'Fase 2'), (16, ':', 'Fase 3')]:
    if xline < len(history['train_loss']):
        axes[0].axvline(x=xline, color='gray', linestyle=ls, alpha=0.5, label=label)
axes[0].set_title('Curvas de Pérdida', fontweight='bold')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, history['f1_comb'], 'g-o', markersize=4, label='F1 Combinado')
axes[1].plot(epochs_range, history['f1_l1'],   'b--', markersize=3, label='F1 L1', alpha=0.7)
axes[1].plot(epochs_range, history['f1_l2'],   'r--', markersize=3, label='F1 L2', alpha=0.7)
if history['f1_comb']:
    best_ep = np.argmax(history['f1_comb'])
    axes[1].axvline(x=best_ep+1, color='gold', linewidth=2,
                    label=f'Mejor: {max(history["f1_comb"]):.4f}')
axes[1].set_title(f'Macro F1 — {CFG["backbone"]}', fontweight='bold')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Macro F1')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(DRIVE_DIR, f'curvas_{CFG["backbone"].replace("/","_")}_fold{CFG["fold"]}.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f'Gráfico guardado en Drive: {plot_path}')
plt.show()

In [ ]:
# ─── Celda 13: Generación de Submission ──────────────────────────────────────
@torch.no_grad()
def generate_submission(model, test_df, test_dir,
                         output_path=None):
    if output_path is None:
        name = CFG['backbone'].replace('/', '_')
        output_path = os.path.join(DRIVE_DIR,
                                    f'submission_{name}_fold{CFG["fold"]}.csv')
    model.eval()
    test_ds     = FoodDataset(test_df, test_dir, get_val_transforms(), is_test=True)
    test_loader = DataLoader(test_ds, batch_size=CFG['batch_size'] * 2,
                             shuffle=False, num_workers=CFG['num_workers'])

    filenames, preds_l1, preds_l2 = [], [], []
    for imgs, fnames in test_loader:
        imgs = imgs.to(DEVICE)
        out_l1, out_l2 = model(imgs)
        filenames.extend(fnames)
        preds_l1.extend(out_l1.argmax(1).cpu().numpy())
        preds_l2.extend((out_l2.argmax(1) + 6).cpu().numpy())

    labels = [f'{p1} {p2}' for p1, p2 in zip(preds_l1, preds_l2)]
    sub_df = pd.DataFrame({'filename': filenames, 'label': labels})
    sub_df.to_csv(output_path, index=False)
    print(f'Submission guardada: {output_path}')
    print(sub_df.head(5))
    return sub_df


sub_df = generate_submission(model, test_df, TEST_DIR)

In [ ]:
# ─── Celda 14: Guía para cambiar de backbone ─────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════════╗
║        GUÍA PARA ESCALAR A OTROS BACKBONES                      ║
╠══════════════════════════════════════════════════════════════════╣
║  1. Cambia CFG['backbone'] en la Celda 4                        ║
║  2. Ajusta CFG['batch_size'] si hay OOM (Out Of Memory)         ║
║  3. El checkpoint es independiente por backbone                 ║
║     → cada backbone tiene su propio archivo en Drive            ║
║  4. Re-ejecuta TODAS las celdas (Entorno de ejecución →         ║
║     Reiniciar y ejecutar todo)                                  ║
║                                                                 ║
║  BACKBONES RECOMENDADOS (Colab Pro, GPU T4):                    ║
║   • 'efficientnet_b0'    → Línea base rápida                    ║
║   • 'efficientnet_b3'    → Balance velocidad/accuracy           ║
║   • 'convnext_small'     → Moderno, buen accuracy               ║
║   • 'swin_small_patch4_window7_224' → Vision Transformer        ║
║   • 'resnet50'           → Clásico robusto                      ║
║                                                                 ║
║  Si la sesión se interrumpe:                                    ║
║   1. Reconectar a Colab                                         ║
║   2. Entorno de ejecución → Reiniciar y ejecutar todo           ║
║   3. El notebook detecta el checkpoint y continúa solo          ║
╚══════════════════════════════════════════════════════════════════╝
""")